# 08 Custom Chattiness Evaluator

This notebook defines a custom evaluator for response chattiness, loads prior JSON output, and batch-scores both model responses per pair.

The output is displayed using the same renderer style as Notebook 05.

## Load Prior Output

In [ ]:
import json
from pathlib import Path

pairs_output_path = Path('../outputs/similarity-pairs-results.json')
if not pairs_output_path.exists():
    raise RuntimeError('Missing ../outputs/similarity-pairs-results.json. Run 07_pairwise_similarity.ipynb first.')

rows = json.loads(pairs_output_path.read_text(encoding='utf-8'))
print(f'Loaded {len(rows)} rows from {pairs_output_path}')

required = {'pair', 'query', 'response_4o', 'response_51'}
missing = [i + 1 for i, r in enumerate(rows) if not required.issubset(r.keys())]
if missing:
    raise RuntimeError(f'Rows missing required fields at indices: {missing}')

rows[0].keys()

## Define Custom Evaluator

In [ ]:
import re
from dataclasses import dataclass

@dataclass
class ChattinessResult:
    score: float
    words: int
    sentences: int
    avg_words_per_sentence: float
    filler_hits: int
    reason: str

class ChattinessEvaluator:
    """Simple deterministic evaluator where higher score means more verbose/chattier output."""

    filler_terms = [
        'basically', 'actually', 'just', 'really', 'very', 'in general',
        'for example', 'in practice', 'overall', 'typically'
    ]

    def evaluate(self, text: str) -> ChattinessResult:
        clean = (text or '').strip()
        words = len(re.findall(r"\b\w+[\w'-]*\b", clean))
        sentences = max(1, len([s for s in re.split(r"[.!?]+", clean) if s.strip()]))
        avg_wps = words / sentences if sentences else float(words)

        lower = clean.lower()
        filler_hits = sum(lower.count(term) for term in self.filler_terms)

        # Calibrated to avoid ceiling effects and preserve model-to-model spread.
        # Typical score band in this demo should be around 2.0 - 4.5 rather than saturating at 5.
        raw = 1.0 + (words / 55.0) + (avg_wps / 18.0) + (filler_hits * 0.2)
        score = round(min(5.0, max(1.0, raw)), 2)

        reason = (
            f'words={words}, sentences={sentences}, avg_words/sentence={avg_wps:.1f}, '
            f'filler_hits={filler_hits}, raw={raw:.2f}'
        )

        return ChattinessResult(
            score=score,
            words=words,
            sentences=sentences,
            avg_words_per_sentence=round(avg_wps, 2),
            filler_hits=filler_hits,
            reason=reason,
        )

## Batch Evaluate Both Models

In [ ]:
evaluator = ChattinessEvaluator()

chattiness_rows = []
for row in rows:
    r4o = evaluator.evaluate(row['response_4o'])
    r51 = evaluator.evaluate(row['response_51'])

    chattier = 'gpt-4o' if r4o.score > r51.score else 'gpt-5.1' if r51.score > r4o.score else 'tie'

    chattiness_rows.append({
        'pair': row['pair'],
        'query': row['query'],
        'response_4o': row['response_4o'],
        'response_51': row['response_51'],
        'chattiness_4o': r4o.score,
        'chattiness_51': r51.score,
        'words_4o': r4o.words,
        'words_51': r51.words,
        'sentences_4o': r4o.sentences,
        'sentences_51': r51.sentences,
        'avg_wps_4o': r4o.avg_words_per_sentence,
        'avg_wps_51': r51.avg_words_per_sentence,
        'filler_hits_4o': r4o.filler_hits,
        'filler_hits_51': r51.filler_hits,
        'reason_4o': r4o.reason,
        'reason_51': r51.reason,
        'chattier_model': chattier,
    })

print(f'Batch-evaluated {len(chattiness_rows)} rows.')
for r in chattiness_rows:
    print(
        f"Pair {r['pair']}: "
        f"4o={r['chattiness_4o']} (w={r['words_4o']}, s={r['sentences_4o']}) vs "
        f"5.1={r['chattiness_51']} (w={r['words_51']}, s={r['sentences_51']}) -> {r['chattier_model']}"
    )

## Display In Notebook 05 Style

In [ ]:
import importlib
import sys
from IPython.display import Markdown, display

app_dir = Path('../app').resolve()
if str(app_dir) not in sys.path:
    sys.path.append(str(app_dir))

import comparison_renderer
importlib.reload(comparison_renderer)

avg_4o = sum(r['chattiness_4o'] for r in chattiness_rows) / len(chattiness_rows)
avg_51 = sum(r['chattiness_51'] for r in chattiness_rows) / len(chattiness_rows)
avg_delta = avg_51 - avg_4o

if avg_delta > 0:
    overall = 'gpt-5.1 is chattier on average.'
elif avg_delta < 0:
    overall = 'gpt-4o is chattier on average.'
else:
    overall = 'Both are equally chatty on average.'

# Show the key result in a prominent summary block before the detailed cards.
display(Markdown(
    '## Key Result\n'
    f'**{overall}**\n\n'
    f'- Average chattiness score (gpt-4o): **{avg_4o:.2f}**\n'
    f'- Average chattiness score (gpt-5.1): **{avg_51:.2f}**\n'
    f'- Average delta (5.1 - 4o): **{avg_delta:+.2f}**'
))

render_rows = []
for r in chattiness_rows:
    diff = r['chattiness_51'] - r['chattiness_4o']
    winner = 'gpt-5.1' if diff > 0 else 'gpt-4o' if diff < 0 else 'tie'

    word_delta = r['words_51'] - r['words_4o']
    sent_delta = r['sentences_51'] - r['sentences_4o']
    avg_wps_delta = r['avg_wps_51'] - r['avg_wps_4o']
    filler_delta = r['filler_hits_51'] - r['filler_hits_4o']

    if winner == 'tie':
        headline = 'Result: both models are equally chatty on this prompt.'
    else:
        headline = f"Result: {winner} is chattier by {abs(diff):.2f} points on this prompt."

    eval_note = (
        f"{headline}\n"
        f"Score comparison: gpt-4o={r['chattiness_4o']} vs gpt-5.1={r['chattiness_51']} (delta={diff:+.2f})\n"
        f"Why: word delta={word_delta:+d}, sentence delta={sent_delta:+d}, "
        f"avg words/sentence delta={avg_wps_delta:+.2f}, filler delta={filler_delta:+d}.\n"
        f"gpt-4o -> words={r['words_4o']}, sentences={r['sentences_4o']}, "
        f"avg words/sentence={r['avg_wps_4o']}, filler hits={r['filler_hits_4o']}\n"
        f"gpt-5.1 -> words={r['words_51']}, sentences={r['sentences_51']}, "
        f"avg words/sentence={r['avg_wps_51']}, filler hits={r['filler_hits_51']}"
    )

    render_rows.append({
        'scenario_id': f"pair_{r['pair']:02d}",
        'category': 'Custom Evaluator: Chattiness',
        'system_label': 'Custom evaluator',
        'system': 'ChattinessEvaluator (higher = more verbose)',
        'user_label': 'Prompt',
        'user': r['query'],
        'difference_label': 'Custom evaluation summary',
        'difference_explanation': eval_note,
        'badges': [],
        'primary_deployment': 'chat4o',
        'secondary_deployment': 'chat51',
        'primary_status': 'success',
        'secondary_status': 'success',
        'primary_response': r['response_4o'],
        'secondary_response': r['response_51'],
    })

comparison_renderer.render_comparison(render_rows)

## Save Custom Evaluator Output

In [ ]:
custom_output_path = Path('../outputs/custom-chattiness-results.json')
custom_output_path.write_text(json.dumps(chattiness_rows, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Saved custom evaluator results to {custom_output_path}')